In [ ]:
import os
import json
import time # 서버에 있는 시간을 기록

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from dotenv import load_dotenv

In [ ]:
from google.colab import userdata
import os

# Colab Secrets에서 'OPENAI_API_KEY'를 가져옵니다.
try:
    api_key = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = api_key
    print(f'Successfully loaded API key: {api_key[:10]}...')
except Exception as e:
    print('Colab Secrets에서 OPENAI_API_KEY를 찾을 수 없습니다. 왼쪽 열쇠 아이콘에서 설정을 확인해주세요.')

Successfully loaded API key: sk-proj-__...


`.env` 파일 작성

In [ ]:
# Create a .env file with a placeholder for the OpenAI API key
with open('.env', 'w') as f:
    f.write('OPENAI_API_KEY="YOUR_OPENAI_API_KEY"\n')

print('.env file created successfully.')

.env file created successfully.


`env 파일의 API 키 로딩`

In [ ]:
from dotenv import load_dotenv
import os

# Load the environment variables from the .env file
load_dotenv() # This searches for a .env file in the current directory

# Retrieve the API key
api_key = os.getenv('OPENAI_API_KEY')

if api_key:
    print(f'Successfully loaded API key: {api_key[:10]}...')
else:
    print('Failed to load API key.')

Successfully loaded API key: sk-proj-__...


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

## 임베딩이란

In [ ]:
# 이미지, 영상, 텍스트, 음성 등 -> 벡터 공간에 매핑하는 것
# 고차원 데이터의 저차원 압축 표현
# 예: (3, 1024, 1024) -> (32, 32)

In [ ]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# API 키가 환경 변수에 설정되어 있는지 확인 후 모델 재생성
llm = ChatOpenAI(model='gpt-4o-mini')
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')

print("모델이 새로운 API 키로 재설정되었습니다.")

모델이 새로운 API 키로 재설정되었습니다.


In [ ]:
test_emb = embeddings_model.embed_query("Hello")

In [ ]:
test_emb[:10]

[0.0192108154296875,
 -0.06451416015625,
 -0.0017137527465820312,
 0.07806396484375,
 0.0215911865234375,
 -0.0155487060546875,
 -0.0150299072265625,
 0.0457763671875,
 -0.005931854248046875,
 -0.0452880859375]

In [ ]:
import tiktoken

1. **tiktoken**: OpenAI 모델이 사용하는 고성능 토크나이저 라이브러리입니다.
2. **encoding_for_model**: 특정 모델의 규칙에 맞게 텍스트를 토큰(숫자)으로 변환하는 설정을 불러옵니다.

In [ ]:
enc = tiktoken.encoding_for_model('gpt-4o-mini')

In [ ]:
text = "안녕하세요, 오늘 LLM에 대해 배워보겠습니다."
tokens = enc.encode(text)

In [ ]:
tokens

[14307,
 171731,
 11,
 106820,
 451,
 19641,
 3107,
 67946,
 33628,
 33771,
 8122,
 105216,
 13]

In [ ]:
enc.decode(tokens)

'안녕하세요, 오늘 LLM에 대해 배워보겠습니다.'

In [ ]:
# text => tokenize [] =>[14307, 171731, 11,...]

In [ ]:
# 토큰마다 내용 확인
# 토큰(1M당) 사용량 가격표의 토큰이 바로 이것이라 생각.
for i, token_id in enumerate(tokens):
  token_text = enc.decode([token_id])
  print(f"token{i+1} :  ID:{token_id} -> {token_text}")

token1 :  ID:14307 -> 안
token2 :  ID:171731 -> 녕하세요
token3 :  ID:11 -> ,
token4 :  ID:106820 ->  오늘
token5 :  ID:451 ->  L
token6 :  ID:19641 -> LM
token7 :  ID:3107 -> 에
token8 :  ID:67946 ->  대해
token9 :  ID:33628 ->  배
token10 :  ID:33771 -> 워
token11 :  ID:8122 -> 보
token12 :  ID:105216 -> 겠습니다
token13 :  ID:13 -> .


In [ ]:
# 왜 token1은 "안" 만 있고, token2는 "녕하세요"?
# LLM tokenizing 방식에 의한 것

# <초창기>
# I like go to school : I, like, go, to, school
# He likes go to school ??? like <-> likes (OOV: Out of Vocabulary)

# <현재>
# BPE: Byte Pair Encoding 사용
# 처음에는 쪼갰다가 "안" "녕" "하" "세" "요" -> for loop를 돌면서 많이/자주 나오는 것들을 붙여버림
# 안녕 하 세 요 - 안녕 하 세요 -> 안녕 하세요
# likes = like + s

## 텍스트 분할 (Text Splitting)

### 텍스트 분할기(Text Splitter) 설명

1. **CharacterTextSplitter**: 지정한 글자 수나 줄바꿈(`\n\n`)이 나오면 **기계적으로 싹둑** 자릅니다. 문장의 중간이 잘릴 수도 있어 문맥이 깨지기 쉽습니다.
2. **RecursiveCharacterSplitter**: 가장 많이 쓰입니다. 먼저 단락(`\n\n`) 단위로 보고, 너무 크면 문장(`\n`), 그다음은 단어(` `) 순으로 **최대한 의미가 통하는 덩어리**가 되도록 유연하게 자릅니다.
3. **from_tiktoken_encoder**: 글자 수가 아니라 **AI가 읽는 토큰(Token) 개수**를 기준으로 자릅니다. AI 모델이 한 번에 읽을 수 있는 양이 정해져 있기 때문에 '용량 초과'를 막기에 가장 정확합니다.

In [ ]:
# 텍스트 또는 문서들을 chunk 단위로 나눔
# Spliter 등을 제공하며 이는 chunk 단위로 나누어주는 클래스

# CharacterTextSplitter
# RecursiveCharacterSplitter
# from_tiktoken_encoder()

In [ ]:
long_text = """인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다."""

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20, separators=["\n\n", "\n", ".",""])

# chunk_size: 하나의 청크에 들어가는 글자들의 길이
# chunk_overlap: 문장을 칼같이 자르지 않고, 중첩되는 문장들(단어들)을 청크에 남겨놓는 것 (맥락 파악하도록)
# separators: 어디에서 나누면 될지에 대한 우선순위를 정해줌 (엔터가 많이 나오는곳 또는 공백, 마침표 등등. 내가 다루는 문자열 데이터의 특성에 따라 조절)

In [ ]:
chunks = splitter.split_text(long_text)

In [ ]:
len(chunks)

7

In [ ]:
for i, chunk in enumerate(chunks):
  print(f"chunk {i} : {len(chunk)} characters.")
  print(chunk) # chunk_size= 100으로 줬는데 왜 청크들 사이즈는 65, 46, 등.. 100 이하?
                  # RecursiveCharacterTextSplitter의 로직때문임 (최대한 의미가 통하는 덩어리끼리 자름)
                  # 문장이 정상적으로 끝나는 것으로 판단 되면, 중첩(overlap)을 안하는 것으로 보임
                  # chunk_size는 모델의 컨텍스트 윈도우와 관련

chunk 0 : 65 characters.
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
chunk 1 : 46 characters.
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다
chunk 2 : 65 characters.
. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.
chunk 3 : 46 characters.
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다
chunk 4 : 94 characters.
. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
chunk 5 : 40 characters.
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다
chunk 6 : 61 characters.
. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10, separators=["\n\n", "\n", ".",""])
chunks = splitter.split_text(long_text)

In [ ]:
for i, chunk in enumerate(chunks):
  print(f"chunk {i} : {len(chunk)} characters.")
  print(chunk) # chink_size= 50으로 하니, 중첩이 보이며 중간에서 끊기는 청크들이 보임

chunk 0 : 50 characters.
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적
chunk 1 : 23 characters.
지각능력을 인공적으로 구현하려는 기술입니다
chunk 2 : 1 characters.
.
chunk 3 : 46 characters.
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다
chunk 4 : 50 characters.
. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인
chunk 5 : 23 characters.
분야에서 혁신적인 성과를 보여주고 있습니다
chunk 6 : 1 characters.
.
chunk 7 : 46 characters.
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다
chunk 8 : 50 characters.
. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retri
chunk 9 : 49 characters.
RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용
chunk 10 : 12 characters.
구축에 특히 유용합니다
chunk 11 : 1 characters.
.
chunk 12 : 40 characters.
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다
chunk 13 : 49 characters.
. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여
chunk 14 : 19 characters.
청크를 검색하여 LLM에 전달합니다
chunk 15 : 1 characters.
.


#### tiktoken 기반으로 청크 나누기
- 기존: RecursiveCharacterTextSplitter만 사용
- 이후: RecursiveCharacterTextSplitter + .from_tiktoken_encoder

In [ ]:
splitter_tiktoken = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name = 'gpt-4o-mini',
    chunk_size=50,
    chunk_overlap=10,
)

In [ ]:
def count_token(text, model = 'gpt-4o-mini'):
  enc = tiktoken.encoding_for_model(model)
  tokens = enc.encode(text)
  return len(tokens)

chunks = splitter_tiktoken.split_text(long_text)
for i, chunk in enumerate(chunks):
  tokens = count_token(chunk)
  print(f"chunk {i} : **{len(chunk)} characters**, **{tokens} tokens**.")
  print(chunk)
 # chunk_size = 50인데 왜 65, 100 characters냐? => `글자` 가 아니라 `토큰수` 이기 때문임

chunk 0 : **65 characters**, **40 tokens**.
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
chunk 1 : **100 characters**, **47 tokens**.
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를
chunk 2 : **27 characters**, **15 tokens**.
처리 분야에서 혁신적인 성과를 보여주고 있습니다.
chunk 3 : **80 characters**, **47 tokens**.
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을
chunk 4 : **72 characters**, **27 tokens**.
벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
chunk 5 : **84 characters**, **49 tokens**.
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를
chunk 6 : **31 characters**, **17 tokens**.
후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.


In [ ]:
test_text = """서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다."""

In [ ]:
# 실습: test_text를 토큰 기준으로 스플릿 하고 chunking 하세요, 다양한 chunk_size, overlap size를 이용하세요.

# 스플리터, 틱토큰
splitter_tiktoken = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name = 'gpt-4o-mini',
    chunk_size=100,
    chunk_overlap=20,
)

chunks = splitter_tiktoken.split_text(test_text)

In [ ]:
chunks

['서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한',
 '랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다.']

In [ ]:
for i, chunk in enumerate(chunks):
  tokens = count_token(chunk)
  print(f"chunk {i} : **{len(chunk)} characters**, **{tokens} tokens**.")
  print(chunk)

chunk 0 : **152 characters**, **97 tokens**.
서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한
chunk 1 : **92 characters**, **55 tokens**.
랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다.


In [ ]:
def split_and_report(text, chunk_size=50,chunk_overlap=10):
  splitter_tiktoken = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name = 'gpt-4o-mini',
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
  )
  chunks = splitter.split_text(text)
  token_counts = [count_token(c) for c in chunks]

  print(f"chunking report (chunk_size = {chunk_size}, chunk_overlap ={chunk_overlap})")
  for i, (chunk, tc) in enumerate(zip(chunks, token_counts)):
    first_line = chunk.split("\n")[0][:40]
    print(f"chunk {i+1} : **{len(chunk)} characters**, **{tc} tokens**, **{first_line}...**")

  print(f"summary: {len(chunks)} chunks")
  print(f"avg: {np.mean(token_counts)} tokens per chunk")
  print(f"max: {max(token_counts)} tokens")

In [ ]:
split_and_report(test_text)

chunking report (chunk_size = 50, chunk_overlap =10)
chunk 1 : **72 characters**, **42 tokens**, **서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북...**
chunk 2 : **99 characters**, **65 tokens**, **. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워...**
chunk 3 : **40 characters**, **26 tokens**, **. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다....**
summary: 3 chunks
avg: 44.333333333333336 tokens per chunk
max: 65 tokens


In [ ]:
# 지금까지 한건 거대한 문서를 잘라주는것

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
embedding_model = OpenAIEmbeddings(model = 'text-embedding-3-small')
embedding_model("hello")

TypeError: 'OpenAIEmbeddings' object is not callable

In [ ]:
query = " 인공지능이 세상을 바꾸고 있습니다."
query_vector = embedding_model.embed_query(query)

In [ ]:
len(query_vector) #임베딩 차원은 모델이 결정함

1536

## 유사도 계산과 벡터 검색

### RAG(Retrieval-Augmented Generation) 작동 순서

우리가 모델에 질문을 했을 때, 내부적으로 일어나는 과정은 다음과 같습니다.

1. **질문 임베딩 (Text → Vector)**
    * 사용자의 질문(텍스트)을 임베딩 모델을 통해 **벡터(숫자 리스트)**로 변환합니다.

2. **유사도 검색 (Vector Search)**
    * 질문 벡터와 미리 저장된 문서 청크 벡터들을 비교하여, 가장 관련성이 높은 **텍스트 조각(청크)**들을 찾아냅니다.

3. **프롬프트 구성 (Vector → Text 합치기) ★ 중요**
    * 검색된 결과는 다시 **텍스트 형태**로 꺼내어집니다.
    * 이 '참고 지식'과 '사용자의 질문'을 하나의 자연어 문장으로 합칩니다.
    * **예시:** "[지식: 서울은 대한민국의 수도이다.] 위 내용을 참고해서 '한국의 수도는 어디인가요?'라는 질문에 답하세요."

4. **LLM 최종 답변 생성**
    * 구성된 전체 텍스트 프롬프트를 GPT-4o 같은 모델에 전달하고, 모델은 이를 읽고 최종 답변을 내놓습니다.

**결론:** 벡터는 방대한 데이터 속에서 **바늘(관련 정보)을 찾기 위한 자석** 같은 역할이며, 실제 대화는 다시 **텍스트(말소리)**로 이루어집니다.

In [ ]:
texts = [
    "AI가 세상을 바꾸고 있다.",
    "AI가 의료분야를 혁신하고 있다.",
    "머신러닝으로 질병을 예측할 수 있다.",
    "오늘 저녁에 치킨을 시켜먹었다."
]

In [ ]:
doc_vectors = embedding_model.embed_documents(texts)

In [ ]:
len(doc_vectors)

4

In [ ]:
for vec in doc_vectors:
  print(len(vec))

1536
1536
1536
1536


In [ ]:
doc_vectors[0][:5], doc_vectors[1][:5]

([0.02008056640625,
  0.0251007080078125,
  -0.00824737548828125,
  0.0438232421875,
  -0.0008625984191894531],
 [-0.039520263671875,
  -0.0017099380493164062,
  0.01319122314453125,
  0.0625,
  0.00557708740234375])

In [ ]:
# 벡터간 유사도 -> 코사인 유사도
# 두 벡터간의 사잇각을 체크

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity(doc_vectors) # 1은 자기 자신과의 유사도

# texts 리스트의 0, 1, 2, 3번째 요소들

array([[1.        , 0.48046445, 0.16744409, 0.15169336],
       [0.48046445, 1.        , 0.28594307, 0.15945624],
       [0.16744409, 0.28594307, 1.        , 0.09681747],
       [0.15169336, 0.15945624, 0.09681747, 1.        ]])

In [ ]:
texts = [
    "AI가 세상을 바꾸고 있다.",
    "AI가 의료분야를 혁신하고 있다.",
    "머신러닝으로 질병을 예측할 수 있다.",
    "오늘 저녁에 치킨을 시켜먹었다."
]

In [ ]:
doc_vectors = embedding_model.embed_documents(texts)

In [ ]:
query_vector = embedding_model.embed_query(query)

In [ ]:
all_vectors = [query_vector] + doc_vectors
all_texts = [query] + texts

In [ ]:
len(all_vectors)

5

In [ ]:
all_texts

[' 인공지능이 세상을 바꾸고 있습니다.',
 'AI가 세상을 바꾸고 있다.',
 'AI가 의료분야를 혁신하고 있다.',
 '머신러닝으로 질병을 예측할 수 있다.',
 '오늘 저녁에 치킨을 시켜먹었다.']

In [ ]:
cosine_similarity(all_vectors)

array([[1.        , 0.72027133, 0.30193563, 0.20401313, 0.09856062],
       [0.72027133, 1.        , 0.48046445, 0.16744409, 0.15169336],
       [0.30193563, 0.48046445, 1.        , 0.28594307, 0.15945624],
       [0.20401313, 0.16744409, 0.28594307, 1.        , 0.09681747],
       [0.09856062, 0.15169336, 0.15945624, 0.09681747, 1.        ]])

In [ ]:
similarity_matrix = cosine_similarity(all_vectors)
labels = ["query"] + [f"doc{i+1}" for i in range(len(texts))]

df = pd.DataFrame(similarity_matrix, index = labels, columns = labels)
df

,query,doc1,doc2,doc3,doc4
query,1.000000,0.720271,0.301936,0.204013,0.098561
doc1,0.720271,1.000000,0.480464,0.167444,0.151693
doc2,0.301936,0.480464,1.000000,0.285943,0.159456
doc3,0.204013,0.167444,0.285943,1.000000,0.096817
doc4,0.098561,0.151693,0.159456,0.096817,1.000000


In [ ]:
categories = {
    "기술": ["인공지능과 머신러닝이 산업을 혁신하고 있다", "클라우드 서비스가 기업의 디지털 전환을 가속화한다"],
    "스포츠": ["프로야구 시즌이 시작되어 팬들이 열광하고 있다", "올림픽에서 한국 선수가 금메달을 획득했다"],
    "음식": ["이 레스토랑의 파스타가 정말 맛있었다", "한국의 김치는 세계적으로 유명한 발효 식품이다"],
}

def classify(text, categories):
    text_vector = embedding_model.embed_query(text)

    cat_values = []
    for cat_list in categories.values():
        cat_values.extend(cat_list) # append 대신 extend를 사용하면 리스트 내부의 요소들만 추가

    cat_vectors = embedding_model.embed_documents(cat_values)

    all_vectors = [text_vector] + cat_vectors
    similarity_matrix = cosine_similarity(all_vectors)

    print(pd.DataFrame(similarity_matrix))

    # 첫 번째 행에서 가장 점수가 높은 인덱스 찾기 (0번 자기자신 제외)
    scores_against_docs = similarity_matrix[0, 1:]
    best_doc_idx = np.argmax(scores_against_docs)

    # 어떤 카테고리인지
    num_texts_per_cat = len(list(categories.values())[0])
    best_cat_idx = best_doc_idx // num_texts_per_cat

    category_names = list(categories.keys())
    return category_names[best_cat_idx]

# 테스트
test_input = "고기는 맛있어."
result = classify(test_input, categories)
print(f"입력: {test_input}")
print(f"분류 결과: {result}")

          0         1         2         3         4         5         6
0  1.000000  0.078657  0.046500  0.144922  0.146633  0.365636  0.311599
1  0.078657  1.000000  0.286328  0.192410  0.153487  0.058312  0.217853
2  0.046500  0.286328  1.000000  0.055109  0.085163  0.117924  0.101908
3  0.144922  0.192410  0.055109  1.000000  0.165624  0.202883  0.095725
4  0.146633  0.153487  0.085163  0.165624  1.000000  0.141096  0.224541
5  0.365636  0.058312  0.117924  0.202883  0.141096  1.000000  0.217275
6  0.311599  0.217853  0.101908  0.095725  0.224541  0.217275  1.000000
입력: 고기는 맛있어.
분류 결과: 음식


In [ ]:
import numpy as np

In [ ]:
cat_vectors = {}
for cat, examples in categories.items():
  embs = embedding_model.embed_documents(examples)
  cat_vectors[cat] = np.mean(embs, axis = 0)

In [ ]:
cat_vectors['기술']

array([-0.00663376,  0.02749634,  0.01636887, ..., -0.02396774,
       -0.01489258,  0.02204895])

In [ ]:
# 정답 (강사님 작성)
def classify(text, categories):
  cat_vectors = {}
  for cat, examples in categories.items():
    embs = embedding_model.embed_documents(examples)
    cat_vectors[cat] = np.mean(embs, axis = 0)

  text_emb = embedding_model.embed_query(text)

  scores = {}
  for cat, cat_vec in cat_vectors.items():
    sim = cosine_similarity([text_emb], [cat_vec])[0][0]
    scores[cat] = round(sim, 4)

  best_cat = max(scores, key=scores.get)

  print(f"입력: {text}")
  print(f"예측: {best_cat}")

  for c, s in scores.items():
    marker = " 000" if c == best_cat else ""
    print(f" {c}: {s}{marker}")

  return best_cat, scores

In [ ]:
test_sentences = ["새로운 GPU가 출시되어 AI 학습속도가 빨라졌습니다.", "올해 올림픽에서 한국이 좋은 성적을 거뒀다." ,"이 식당 불고기가 맛있어요."]

for sent in test_sentences:
  print()
  classify(sent, categories)


입력: 새로운 GPU가 출시되어 AI 학습속도가 빨라졌습니다.
예측: 기술
 기술: 0.3744 000
 스포츠: 0.2215
 음식: 0.1131

입력: 올해 올림픽에서 한국이 좋은 성적을 거뒀다.
예측: 스포츠
 기술: 0.1477
 스포츠: 0.5146 000
 음식: 0.2895

입력: 이 식당 불고기가 맛있어요.
예측: 음식
 기술: 0.1205
 스포츠: 0.1822
 음식: 0.4182 000


In [ ]:
# 임베딩은, 동일한 문장을 반복적으로 넣어도 항상 같은 값을 냄
# 이는 운영적 안전성에 도움을 줌 (그냥 LLM을 이용해서 카테고리를 분류하는 것 보다..)

In [ ]:
# 따라서 랭체인에서도 CachedBackedEmbeddings 등의 기능을 지원함
# 벡터들을 캐시에 저장해놨다가 재사용 등

## 임베딩 캐싱 (CacheBackedEmbeddings)

### 임베딩 캐싱 (Embedding Caching)

임베딩은 같은 텍스트에 대해 항상 같은 결과(벡터)를 반환합니다. 따라서 매번 API를 호출하는 대신 결과를 저장해두고 재사용하면 효율적입니다.

1. **CachedBackedEmbeddings**: 임베딩 결과를 특정 저장소(캐시)에 보관하고, 동일한 텍스트가 들어오면 API 호출 없이 즉시 저장된 값을 반환하는 래퍼(Wrapper) 클래스입니다. 이를 통해 API 비용을 절감하고 처리 속도를 높일 수 있습니다.
2. **InMemoryByteStore**: 임베딩 벡터를 서버의 **메모리(RAM)**에 일시적으로 저장하는 저장소입니다. 프로그램이 실행되는 동안 가장 빠르게 데이터를 읽고 쓸 수 있는 공간으로 활용됩니다.

In [ ]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_core.stores import InMemoryByteStore

In [ ]:
pip install langchain_core

In [ ]:
pip install langchain_classic

In [ ]:
store = InMemoryByteStore()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embedding_model, store, namespace = 'embedding-cache')


/usr/local/lib/python3.12/dist-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()
